In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [23]:
train_df = pd.read_csv("/kaggle/input/datasets/shayanfazeli/heartbeat/mitbih_train.csv", header=None)
test_df = pd.read_csv("/kaggle/input/datasets/shayanfazeli/heartbeat/mitbih_test.csv", header=None)

print(train_df.shape)
print(test_df.shape)

(87554, 188)
(21892, 188)


In [24]:
X_train = train_df.iloc[:, :-1].values
y_train = train_df.iloc[:, -1].values

X_test = test_df.iloc[:, :-1].values
y_test = test_df.iloc[:, -1].values

In [25]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [26]:
X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_test = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

print(X_train.shape)

torch.Size([87554, 1, 187])


In [81]:
class ECGDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [109]:
class ECG_CNN(nn.Module):

    def __init__(self):
        super(ECG_CNN, self).__init__()

        self.conv1 = nn.Conv1d(1, 32, kernel_size=5)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=5)
        self.pool2 = nn.MaxPool1d(2)

        self.conv3 = nn.Conv1d(64, 128, kernel_size=3)
        self.pool3 = nn.MaxPool1d(2)

        self.fc1 = nn.Linear(128*20, 128)
        self.fc2 = nn.Linear(128, 5)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):

        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = self.pool3(self.relu(self.conv3(x)))

        x = x.view(x.size(0), -1)

        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)

        return x

In [133]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ECG_CNN().to(device)
print(model)

ECG_CNN(
  (conv1): Conv1d(1, 32, kernel_size=(5,), stride=(1,))
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(32, 64, kernel_size=(5,), stride=(1,))
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv1d(64, 128, kernel_size=(3,), stride=(1,))
  (pool3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=2560, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=5, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.5, inplace=False)
)


In [140]:
from adabelief_pytorch import AdaBelief
criterion = nn.CrossEntropyLoss()
# criterion = nn.CrossEntropyLoss(label_smoothing=0.07)
optimizer = AdaBelief(model.parameters(),lr=0.001)

Please check your arguments if you have upgraded adabelief-pytorch from version 0.0.5.
Modifications to default arguments:
                           eps  weight_decouple    rectify
-----------------------  -----  -----------------  ---------
adabelief-pytorch=0.0.5  1e-08  False              False
>=0.1.0 (Current 0.2.0)  1e-16  True               True
SGD better than Adam (e.g. CNN for Image Classification)    Adam better than SGD (e.g. Transformer, GAN)
----------------------------------------------------------  ----------------------------------------------
Recommended eps = 1e-8                                      Recommended eps = 1e-16
For a complete table of recommended hyperparameters, see
https://github.com/juntang-zhuang/Adabelief-Optimizer
You can disable the log message by setting "print_change_log = False", though it is recommended to keep as a reminder.

Weight decoupling enabled in AdaBelief
Rectification enabled in AdaBelief


In [141]:
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(model.parameters(),lr=0.03)

In [142]:
epochs = 40
best_loss = float("inf")
patience = 3
counter = 0

for epoch in range(epochs):

    # ----- Training Phase -----
    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss = train_loss / len(train_loader)


    # ----- Testing Phase -----
    model.eval()
    test_loss = 0

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            test_loss += loss.item()

    test_loss = test_loss / len(test_loader)

    if test_loss < best_loss:
        best_loss = test_loss
        counter = 0
    else:
        counter += 1
    
    if counter >= patience:
        print("Early stopping triggered")
        break

    # ----- Print Both -----
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")

Epoch [1/40] | Train Loss: 0.0310 | Test Loss: 0.0755
Epoch [2/40] | Train Loss: 0.0263 | Test Loss: 0.0766
Epoch [3/40] | Train Loss: 0.0246 | Test Loss: 0.0821
Early stopping triggered


In [143]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)

        _, predicted = torch.max(outputs, 1)

        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

accuracy = 100 * correct / total

print("Test Accuracy:", accuracy)

Test Accuracy: 98.5337109446373


In [144]:
all_preds = []
all_labels = []

model.eval()

with torch.no_grad():
    
    for X_batch, y_batch in test_loader:
        
        X_batch = X_batch.to(device)
        
        outputs = model(X_batch)
        
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.numpy())

In [145]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[18060    26    19     3    10]
 [  119   425    11     0     1]
 [   47     5  1372    17     7]
 [   23     0    12   127     0]
 [   18     0     3     0  1587]]


In [147]:
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99     18118
           1       0.93      0.76      0.84       556
           2       0.97      0.95      0.96      1448
           3       0.86      0.78      0.82       162
           4       0.99      0.99      0.99      1608

    accuracy                           0.99     21892
   macro avg       0.95      0.90      0.92     21892
weighted avg       0.98      0.99      0.98     21892

